In [1]:
# SmolLM-135M Implementation (Llama Architecture)
# Based on: https://huggingface.co/HuggingFaceTB/SmolLM-135M

import math
import inspect
from dataclasses import dataclass
from typing import Optional, Tuple

import torch
import torch.nn as nn
from torch.nn import functional as F

# Configuration for SmolLM-135M
@dataclass
class SmolLMConfig:
    block_size: int = 512 # Reduced to 512 for 4GB GPU training
    vocab_size: int = 50304 # Aligned to 50304 for tiktoken compatibility (SmolLM native is 49152)
    n_layer: int = 30
    n_head: int = 9
    n_kv_head: int = 3 # Grouped Query Attention (GQA)
    n_embd: int = 576
    intermediate_size: int = 1536 # SwiGLU intermediate size
    rms_norm_eps: float = 1e-5
    rope_theta: float = 10000.0
    dropout: float = 0.0
    bias: bool = False # True: bias in Linears and LayerNorms, like GPT-2. False: a bit better and faster

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        output = self._norm(x.float()).type_as(x)
        return output * self.weight

def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)  # complex64
    return freqs_cis

def reshape_for_broadcast(freqs_cis: torch.Tensor, x: torch.Tensor):
    ndim = x.ndim
    assert 0 <= 1 < ndim
    assert freqs_cis.shape == (x.shape[1], x.shape[-1])
    shape = [d if i == 1 or i == ndim - 1 else 1 for i, d in enumerate(x.shape)]
    return freqs_cis.view(*shape)

def apply_rotary_emb(xq: torch.Tensor, xk: torch.Tensor, freqs_cis: torch.Tensor):
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = reshape_for_broadcast(freqs_cis, xq_)
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    return xq_out.type_as(xq), xk_out.type_as(xk)

class CausalSelfAttention(nn.Module):
    def __init__(self, config: SmolLMConfig):
        super().__init__()
        self.n_head = config.n_head
        self.n_kv_head = config.n_kv_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        self.n_rep = self.n_head // self.n_kv_head

        self.wq = nn.Linear(config.n_embd, config.n_head * self.head_dim, bias=config.bias)
        self.wk = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=config.bias)
        self.wv = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=config.bias)
        self.wo = nn.Linear(config.n_head * self.head_dim, config.n_embd, bias=config.bias)

        self.dropout = config.dropout
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor, freqs_cis: torch.Tensor):
        B, T, C = x.shape

        xq, xk, xv = self.wq(x), self.wk(x), self.wv(x)
        xq = xq.view(B, T, self.n_head, self.head_dim)
        xk = xk.view(B, T, self.n_kv_head, self.head_dim)
        xv = xv.view(B, T, self.n_kv_head, self.head_dim)

        xq, xk = apply_rotary_emb(xq, xk, freqs_cis=freqs_cis)

        # Grouped Query Attention: repeat k/v heads to match q heads
        xk = torch.repeat_interleave(xk, dim=2, repeats=self.n_rep)
        xv = torch.repeat_interleave(xv, dim=2, repeats=self.n_rep)

        # Make heads batch dimension
        xq = xq.transpose(1, 2)  # (B, n_head, T, head_dim)
        xk = xk.transpose(1, 2)  # (B, n_head, T, head_dim)
        xv = xv.transpose(1, 2)  # (B, n_head, T, head_dim)

        # Flash Attention
        output = F.scaled_dot_product_attention(xq, xk, xv, is_causal=True)

        output = output.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.wo(output))

class SwiGLU(nn.Module):
    def __init__(self, config: SmolLMConfig):
        super().__init__()
        self.w1 = nn.Linear(config.n_embd, config.intermediate_size, bias=config.bias) # Gate
        self.w3 = nn.Linear(config.n_embd, config.intermediate_size, bias=config.bias) # Value
        self.w2 = nn.Linear(config.intermediate_size, config.n_embd, bias=config.bias) # Output
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))

class Block(nn.Module):
    def __init__(self, config: SmolLMConfig):
        super().__init__()
        self.attention_norm = RMSNorm(config.n_embd, eps=config.rms_norm_eps)
        self.attention = CausalSelfAttention(config)
        self.ffn_norm = RMSNorm(config.n_embd, eps=config.rms_norm_eps)
        self.feed_forward = SwiGLU(config)

    def forward(self, x: torch.Tensor, freqs_cis: torch.Tensor):
        h = x + self.attention(self.attention_norm(x), freqs_cis)
        out = h + self.feed_forward(self.ffn_norm(h))
        return out

class SmolLM(nn.Module):
    def __init__(self, config: SmolLMConfig):
        super().__init__()
        self.config = config
        self.tok_embeddings = nn.Embedding(config.vocab_size, config.n_embd)
        self.layers = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.norm = RMSNorm(config.n_embd, eps=config.rms_norm_eps)
        self.output = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Weight sharing
        self.tok_embeddings.weight = self.output.weight

        # Precompute RoPE frequencies
        self.freqs_cis = precompute_freqs_cis(config.n_embd // config.n_head, config.block_size * 2, config.rope_theta)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_embeddings(idx)
        
        # Ensure freqs_cis is on the correct device
        if self.freqs_cis.device != x.device:
            self.freqs_cis = self.freqs_cis.to(x.device)
        freqs_cis = self.freqs_cis[:T]

        for layer in self.layers:
            x = layer(x, freqs_cis)
        
        x = self.norm(x)
        logits = self.output(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        
        return logits, loss

# Device selection
device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
print(f"using device: {device}")

# Data Loading
import tiktoken

class DataLoaderLite:
    def __init__(self, B, T):
        self.B = B
        self.T = T

        # Load tokens from disk
        try:
            with open('input.txt', 'r', encoding='utf-8') as f:
                text = f.read()
        except FileNotFoundError:
            print("Error: input.txt not found. Please ensure the file exists.")
            text = "Hello world " * 1000 # Fallback for testing if file missing
            
        enc = tiktoken.get_encoding('gpt2') 
        tokens = enc.encode(text)
        self.tokens = torch.tensor(tokens)
        print(f'loaded {len(self.tokens)} tokens')
        print(f'1 epoch = {len(self.tokens) // (B * T)} batches')

        self.current_position = 0
    
    def next_batch(self):
        B, T = self.B, self.T
        buf = self.tokens[self.current_position: self.current_position + B * T + 1]
        x = (buf[:-1]).view(B, T) # inputs
        y = (buf[1:]).view(B, T) # targets
        self.current_position += B*T
        if self.current_position + (B * T + 1) > len(self.tokens):
            self.current_position = 0
        return x, y

# Training Setup
torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed(1337)

torch.set_float32_matmul_precision('high')

config = SmolLMConfig()
model = SmolLM(config)
model.to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

# Generation Function
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, top_k=None):
    """
    Take a conditioning sequence of indices idx (LongTensor of shape (b,t)) and complete
    the sequence max_new_tokens times, feeding the predictions back into the model each time.
    """
    for _ in range(max_new_tokens):
        # if the sequence context is growing too long we must crop it at block_size
        idx_cond = idx if idx.size(1) <= model.config.block_size else idx[:, -model.config.block_size:]
        # forward the model to get the logits for the index in the sequence
        logits, _ = model(idx_cond)
        # pluck the logits at the final step and scale by desired temperature
        logits = logits[:, -1, :] / temperature
        # optionally crop the logits to only the top k options
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')
        # apply softmax to convert logits to (normalized) probabilities
        probs = F.softmax(logits, dim=-1)
        # sample from the distribution
        idx_next = torch.multinomial(probs, num_samples=1)
        # append sampled index to the running sequence and continue
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

# Training Loop
train_loader = DataLoaderLite(B = 4, T = 512) # Reduced batch size and context for 4GB GPU
optimizer = torch.optim.AdamW(model.parameters(), lr = 3e-4)

import time
import os

max_steps = 5000
eval_interval = 500
save_path = "smollm_135_checkpoint.pth"

print("Starting training...")
for i in range(max_steps):
    t0 = time.time()
    x, y = train_loader.next_batch()
    x, y = x.to(device), y.to(device)
    optimizer.zero_grad()
    
    # Mixed precision training
    with torch.autocast(device_type=device, dtype=torch.bfloat16 if device=='cuda' else torch.float32):
        logits, loss = model(x, y) 
    
    loss.backward()
    optimizer.step()
    
    if device == 'cuda':
        torch.cuda.synchronize() 
        
    t1 = time.time()
    dt = (t1 - t0) * 1000
    tokens_per_sec = (train_loader.B * train_loader.T) / (t1 - t0)
    
    if i % 10 == 0:
        print(f'step {i} | loss: {loss.item():.4f} | dt: {dt:.2f}ms | tok/sec: {tokens_per_sec:.2f}')
        
    # Generate output every 500 steps
    if i > 0 and i % eval_interval == 0:
        print(f"\n--- Generating text at step {i} ---")
        context = torch.zeros((1, 1), dtype=torch.long, device=device) # Start with token 0 (usually valid)
        generated = generate(model, context, max_new_tokens=50)
        # Decode using tiktoken (gpt2 encoding as used in DataLoader)
        enc = tiktoken.get_encoding('gpt2')
        decoded = enc.decode(generated[0].tolist())
        # Force ASCII for Windows console compatibility
        print(decoded.encode('ascii', errors='ignore').decode('ascii'))
        print("-----------------------------------\n")

# Save checkpoint
print(f"Saving model to {save_path}")
checkpoint = {
    'step': max_steps,  # Current step number (5000 in this case)
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'data_loader_position': train_loader.current_position,
    'config': config
}
torch.save(checkpoint, save_path)



using device: cuda
Model parameters: 135.18M
loaded 338025 tokens
1 epoch = 165 batches
Starting training...
step 0 | loss: 10.8725 | dt: 1274.04ms | tok/sec: 1607.49
step 10 | loss: 7.9940 | dt: 2727.77ms | tok/sec: 750.80
step 20 | loss: 6.9461 | dt: 2596.87ms | tok/sec: 788.64
step 30 | loss: 6.7741 | dt: 2602.39ms | tok/sec: 786.97
step 40 | loss: 6.9963 | dt: 2598.25ms | tok/sec: 788.22
step 50 | loss: 6.8732 | dt: 2604.96ms | tok/sec: 786.19
step 60 | loss: 6.5148 | dt: 2597.94ms | tok/sec: 788.32
step 70 | loss: 6.5178 | dt: 2602.95ms | tok/sec: 786.80
step 80 | loss: 6.4811 | dt: 2601.39ms | tok/sec: 787.27
step 90 | loss: 6.5202 | dt: 2616.56ms | tok/sec: 782.71
step 100 | loss: 6.2918 | dt: 2616.17ms | tok/sec: 782.82
step 110 | loss: 6.5149 | dt: 2600.79ms | tok/sec: 787.45
step 120 | loss: 6.1793 | dt: 2644.07ms | tok/sec: 774.56
step 130 | loss: 6.0835 | dt: 2601.48ms | tok/sec: 787.25
step 140 | loss: 5.9245 | dt: 2650.02ms | tok/sec: 772.83
step 150 | loss: 6.4062 | dt: 

In [3]:
# Resume training demonstration
print("\n--- Resuming training from checkpoint ---")

# Load checkpoint
checkpoint = torch.load(save_path, weights_only=False)
start_step = checkpoint['step']  # Get the step we're resuming from
print(f"Resuming from step {start_step}")

# Re-initialize model to prove loading works
model_new = SmolLM(config)
model_new.to(device)
model_new.load_state_dict(checkpoint['model_state_dict'])

# Restore optimizer state
optimizer_new = torch.optim.AdamW(model_new.parameters(), lr = 3e-4)
optimizer_new.load_state_dict(checkpoint['optimizer_state_dict'])

# Restore data loader position
train_loader.current_position = checkpoint['data_loader_position']

print("Checkpoint loaded successfully.")


--- Resuming training from checkpoint ---
Resuming from step 5000
Checkpoint loaded successfully.


In [4]:
# Train for another 50 steps (from step 5001 to 5050)
resume_steps = 50
for i in range(resume_steps):
    current_step = start_step + i + 1  # +1 because we continue from the next step
    t0 = time.time()
    x, y = train_loader.next_batch()
    x, y = x.to(device), y.to(device)
    optimizer_new.zero_grad()
    
    with torch.autocast(device_type=device, dtype=torch.bfloat16 if device=='cuda' else torch.float32):
        logits, loss = model_new(x, y) 
    
    loss.backward()
    optimizer_new.step()
    
    if device == 'cuda':
        torch.cuda.synchronize()
        
    if i % 10 == 0:
        print(f'step {current_step} | loss: {loss.item():.4f}')

print("Resumed training completed.")


step 5001 | loss: 0.0378
step 5011 | loss: 0.0223
step 5021 | loss: 0.0300
step 5031 | loss: 0.0188
step 5041 | loss: 0.0129
Resumed training completed.
